In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys 

sys.path.append(os.path.join(os.getcwd(), '..', '..'))

from utils.transformations import *

###DimUsers

In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/DimUser")

In [0]:
display(df)

####Autoloader


In [0]:
df_user = spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "parquet")\
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimUser/checkpoint")\
        .option("schemaEvolutionMode", "addNewColumns")\
        .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))


In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,["_rescued_data"])
df_user = df_user.dropDuplicates(["user_id"])


In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_data.silver.DimUser")


###DimArtists

In [0]:
df_artists = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "parquet")\
                    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimArtist/checkpoint")\
                    .option("schemaEvolutionMode", "addNewColumns")\
                    .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/DimArtist")


In [0]:
df_art_obj = reusable()

df_artists = df_art_obj.dropColumns(df_artists,["_rescued_data"])
df_artists = df_artists.dropDuplicates(["artist_id"])



In [0]:
df_artists.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_data.silver.DimArtist")


###DimTrack


In [0]:
df_track = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "parquet")\
                    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimTrack/checkpoint")\
                    .option("schemaEvolutionMode", "addNewColumns")\
                    .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/DimTrack")


In [0]:
df_track_obj = reusable()

df_track = df_track.withColumn("duration_flag", when(col("duration_sec") < 150, "low")\
                                    .when(col("duration_sec") <300, "medium")\
                                    .otherwise("high"))

df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))
df_track = df_track_obj.dropColumns(df_track,["_rescued_data"])
df_track = df_track.dropDuplicates(["track_id"])

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_data.silver.DimTrack")


###DimDate


In [0]:
df_date = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "parquet")\
                    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimDate/checkpoint")\
                    .option("schemaEvolutionMode", "addNewColumns")\
                    .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable()

df_date = df_date_obj.dropColumns(df_date,["_rescued_data"])


In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_data.silver.DimDate")


###FactStream


In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                    .option("cloudFiles.format", "parquet")\
                    .option("cloudFiles.schemaLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/FactStream/checkpoint")\
                    .option("schemaEvolutionMode", "addNewColumns")\
                    .load("abfss://bronze@storageazureprojectstef.dfs.core.windows.net/FactStream")

In [0]:
df_fact_obj = reusable()

df_fact = df_fact_obj.dropColumns(df_fact,["_rescued_data"])


In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@storageazureprojectstef.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_data.silver.FactStream")
